# 02 — Ablations: local gains vs public gains

This notebook plots the version history (full table in
`results/versions.md`) and the local-vs-public calibration of `gamma_c`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Version history (local validation RMSE)

Each point is one tested hypothesis; annotations give the main idea. Neutral
versions matter as much as gains: they delimit what is predictable.

In [ ]:
versions = ["v4", "v5.1", "v9", "v10", "v11", "v12", "v13", "v14", "v15"]
local =    [379.8, 375.4, 371.1, 368.2, 348.4, 346.4, 345.2, 345.9, 345.0]
labels = ["probabilistic mixture", "out-of-fold TE", "arrival context",
          "reporting outage", "Rome branch", "5-seed ensembling",
          "per-aircraft signals", "unimpeded time", "off-block queue"]

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(versions, local, marker="o", color="#3B82F6")
for x, y, l in zip(versions, local, labels):
    ax.annotate(l, (x, y), textcoords="offset points", xytext=(4, 7), fontsize=8)
ax.set_ylabel("Validation RMSE (Jan + Jul 2025), s")
ax.set_title("Diminishing returns: −35 s over 11 versions, −20 s from the Rome branch\n"
             "(later removed: −2.45 s on the leaderboard, see versions.md §3.2)")
plt.tight_layout(); plt.show()

## Local vs public calibration of `gamma_c`

Four dedicated submissions (v1-v4): the public curve is parallel to the
local one, with a constant ~32 s offset. Local validation is a reliable
proxy for the physical properties of the problem.

In [ ]:
gammas = [0.5, 0.6, 0.75, 1.0]
loc =    [389.8, 379.4, 380.1, 391.0]
pub =    [358.85, 347.80, 347.59, 355.16]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(gammas, loc, marker="o", label="local validation (Jan+Jul 2025)", color="#3B82F6")
ax.plot(gammas, pub, marker="s", label="leaderboard (Jan+Jul 2026)", color="#10B981")
for g, a, b in zip(gammas, loc, pub):
    ax.annotate(f"{a - b:+.0f} s", (g, (a + b) / 2), fontsize=8, ha="center")
ax.set_xlabel("gamma_c"); ax.set_ylabel("RMSE (s)")
ax.set_title("The local/public offset is constant: local gains transfer")
ax.legend()
plt.tight_layout(); plt.show()

## Public scores

| Submission | Pipeline | Rome branch | Score |
|---|---|---|---|
| v1-v4 | v5.1, gamma sweep | no | 347.59 - 358.85 |
| v7 | v11 | yes | 341.33 |
| v10 | v13 | no | **337.95** |
| v11 | v13 | yes | 340.40 |
| v15 | v15 | yes | 338.02 |

The v10/v11 comparison (same models, only the Rome branch differs) directly
measures the cost of overfitting a bug distribution: **−2.45 s** for a branch
that gained +19.8 s locally.